In [3]:
# plot_level_correlation_with_corners.py
#
# Purpose: Compute Pearson and Spearman correlation between
# labeled plant counts (INCLUDING corner plants) and model detections
# Corner plants were labeled from orthomosaic + validated via directional photos
#
# Field counts source: field_plot_visibility_final.csv
#   n_labeled_total = plants labeled within full tile (3m + corners)
#   n_labeled_deciduous = deciduous labels per plot
#   n_labeled_shrub = shrub labels per plot
#
# Model predictions: saved txt files from model.val(save_txt=True, save_conf=True)
# Confidence threshold: 0.25 (operational inference threshold)

import os
import re
import pandas as pd
import numpy as np
from scipy import stats

# ── paths ──────────────────────────────────────────────────────────────────────
PRED_DIR_SUMMER  = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\field_plot_evaluation_v2\eval_summer\labels"
PRED_DIR_FALL    = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\field_plot_evaluation_v2\eval_fall\labels"
VISIBILITY_CSV = r"C:\Master Physische Geographie\Canada\Masterarbeit\Notes\field_plot_visibility_final.csv"
OUTPUT_DIR       = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\field_plot_evaluation_v2"

CONF_THRESHOLD = 0.25
CLASS_DECIDUOUS = 0
CLASS_SHRUB     = 1

# ── load labeled counts including corners ──────────────────────────────────────
vis = pd.read_csv(VISIBILITY_CSV, sep=";")
vis = vis.sort_values("plot_id").reset_index(drop=True)

print(f"Loaded visibility CSV: {len(vis)} plots")
print(f"Columns: {vis.columns.tolist()}")

# ── load model predictions per plot ───────────────────────────────────────────
def load_predictions(pred_dir, conf_threshold=0.25):
    """Load model detections per plot, filtered by confidence threshold."""
    results = {}
    for fname in sorted(os.listdir(pred_dir)):
        if not fname.endswith(".txt"):
            continue
        # extract plot_id from filename e.g. ranch5_plot00_r023277_c025694.txt
        match = re.search(r"plot(\d+)", fname)
        if not match:
            continue
        plot_id = int(match.group(1))

        n_dec = 0
        n_shr = 0
        with open(os.path.join(pred_dir, fname)) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 6:
                    continue
                cls  = int(parts[0])
                conf = float(parts[5])
                if conf < conf_threshold:
                    continue
                if cls == CLASS_DECIDUOUS:
                    n_dec += 1
                elif cls == CLASS_SHRUB:
                    n_shr += 1

        results[plot_id] = {
            "pred_dec":   n_dec,
            "pred_shr":   n_shr,
            "pred_total": n_dec + n_shr,
        }
    return results

print("\nLoading summer predictions...")
summer_preds = load_predictions(PRED_DIR_SUMMER, CONF_THRESHOLD)
print(f"  Loaded predictions for {len(summer_preds)} plots")

print("Loading fall predictions...")
fall_preds = load_predictions(PRED_DIR_FALL, CONF_THRESHOLD)
print(f"  Loaded predictions for {len(fall_preds)} plots")

# ── build merged dataframe ─────────────────────────────────────────────────────
rows = []
for _, row in vis.iterrows():
    plot_id = int(row["plot_id"])
    sp = summer_preds.get(plot_id, {"pred_dec": 0, "pred_shr": 0, "pred_total": 0})
    fp = fall_preds.get(plot_id,   {"pred_dec": 0, "pred_shr": 0, "pred_total": 0})

    rows.append({
        "plot_id":          plot_id,
        "ranch":            row["ranch"],
        # labeled counts including corners
        "labeled_dec":      row["n_labeled_deciduous"],
        "labeled_shr":      row["n_labeled_shrub"],
        "labeled_total":    row["n_labeled_total"],
        # summer predictions
        "summer_pred_dec":  sp["pred_dec"],
        "summer_pred_shr":  sp["pred_shr"],
        "summer_pred_total":sp["pred_total"],
        # fall predictions
        "fall_pred_dec":    fp["pred_dec"],
        "fall_pred_shr":    fp["pred_shr"],
        "fall_pred_total":  fp["pred_total"],
    })

df = pd.DataFrame(rows).sort_values("plot_id").reset_index(drop=True)
print(f"\nMerged dataframe: {len(df)} plots")

# ── compute correlations ───────────────────────────────────────────────────────
def sig_label(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

results_rows = []

print("\n" + "="*60)
print(f"CORRELATION — labeled counts (incl. corners) vs model detections")
print(f"Confidence threshold: {CONF_THRESHOLD}")
print("="*60)

for season, pred_dec_col, pred_shr_col, pred_tot_col in [
    ("Summer", "summer_pred_dec", "summer_pred_shr", "summer_pred_total"),
    ("Fall",   "fall_pred_dec",   "fall_pred_shr",   "fall_pred_total"),
]:
    print(f"\n--- {season} ---")
    for label, field_col, pred_col in [
        ("Total",        "labeled_total", pred_tot_col),
        ("deciduous",    "labeled_dec",   pred_dec_col),
        ("shrub",        "labeled_shr",   pred_shr_col),
    ]:
        r,   p_r   = stats.pearsonr( df[field_col], df[pred_col])
        rho, p_rho = stats.spearmanr(df[field_col], df[pred_col])
        print(f"  {label:<12}: Pearson r={r:.3f}{sig_label(p_r):<4} "
              f"(p={p_r:.4f}) | Spearman rho={rho:.3f}{sig_label(p_rho):<4} (p={p_rho:.4f})")

        results_rows.append({
            "season":       season,
            "class":        label,
            "pearson_r":    round(r,   3),
            "pearson_p":    round(p_r, 4),
            "pearson_sig":  sig_label(p_r),
            "spearman_rho": round(rho,   3),
            "spearman_p":   round(p_rho, 4),
            "spearman_sig": sig_label(p_rho),
            "field_col":    field_col,
            "n_plots":      len(df),
        })

# ── save results ───────────────────────────────────────────────────────────────
results_df = pd.DataFrame(results_rows)
out_path = os.path.join(OUTPUT_DIR, "plot_level_correlation_with_corners.csv")
results_df.to_csv(out_path, index=False)
print(f"\nSaved: {out_path}")

# save raw data
raw_path = os.path.join(OUTPUT_DIR, "plot_level_data_with_corners.csv")
df.to_csv(raw_path, index=False)
print(f"Saved raw data: {raw_path}")

Loaded visibility CSV: 50 plots
Columns: ['plot_id', 'ranch', 'n_field_3m', 'n_labeled_deciduous', 'n_labeled_shrub', 'n_labeled_coniferous', 'n_labeled_total', 'visibility_pct', 'priority', 'shadow', 'too_small', 'occlusion', 'merged_crowns', 'gps_uncertainty', 'other_reason', 'notes']

Loading summer predictions...
  Loaded predictions for 50 plots
Loading fall predictions...
  Loaded predictions for 50 plots

Merged dataframe: 50 plots

CORRELATION — labeled counts (incl. corners) vs model detections
Confidence threshold: 0.25

--- Summer ---
  Total       : Pearson r=0.850***  (p=0.0000) | Spearman rho=0.906***  (p=0.0000)
  deciduous   : Pearson r=0.803***  (p=0.0000) | Spearman rho=0.886***  (p=0.0000)
  shrub       : Pearson r=0.925***  (p=0.0000) | Spearman rho=0.944***  (p=0.0000)

--- Fall ---
  Total       : Pearson r=0.615***  (p=0.0000) | Spearman rho=0.802***  (p=0.0000)
  deciduous   : Pearson r=0.684***  (p=0.0000) | Spearman rho=0.826***  (p=0.0000)
  shrub       : Pea